In [1]:
import pandas as pd
import json
import os
from google.cloud import bigquery
from google.oauth2 import service_account

# Xử lý file 'transactions_data'

In [2]:
# Đọc file
transactions_df = pd.read_csv('transactions_data.csv')

In [3]:
# Dữ liệu trước khi xử lý
transactions_df.head(5)

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NaN
1,7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,NaN
2,7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,NaN
3,7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,NaN
4,7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,NaN


In [4]:
# Xem qua số cột và dòng trong df
rows, cols = transactions_df.shape
print(f"Kích thước bảng: {rows} dòng và {cols} cột.")

Kích thước bảng: 13305915 dòng và 12 cột.


In [5]:
# Tổng quan về kiểu dữ liệu từng cột
transactions_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13305915 entries, 0 to 13305914
Data columns (total 12 columns):
 #   Column          Dtype  
---  ------          -----  
 0   id              int64  
 1   date            object 
 2   client_id       int64  
 3   card_id         int64  
 4   amount          object 
 5   use_chip        object 
 6   merchant_id     int64  
 7   merchant_city   object 
 8   merchant_state  object 
 9   zip             float64
 10  mcc             int64  
 11  errors          object 
dtypes: float64(1), int64(5), object(6)
memory usage: 1.2+ GB


In [6]:
# Tổng quan các giá trị trùng lặp
trans_duplicate_rows = transactions_df.duplicated().sum()
trans_duplicate_ids = transactions_df['id'].duplicated().sum()
print(f"Số dòng trùng lặp hoàn toàn: {trans_duplicate_rows}")
print(f"Số ID giao dịch bị trùng lặp: {trans_duplicate_ids}")

Số dòng trùng lặp hoàn toàn: 0
Số ID giao dịch bị trùng lặp: 0


In [7]:
# Tổng quan tình trạng Missing value
trans_missing_data = transactions_df.isnull().sum()
trans_missing_percent = (transactions_df.isnull().sum() / rows) * 100
trans_missing_df = pd.DataFrame({'Số lượng Missing': trans_missing_data, 'Tỷ lệ (%)': trans_missing_percent})
print("Tình trạng Missing Value theo từng cột:")
print(trans_missing_df[trans_missing_df['Số lượng Missing'] >= 0])

Tình trạng Missing Value theo từng cột:
                Số lượng Missing  Tỷ lệ (%)
id                             0   0.000000
date                           0   0.000000
client_id                      0   0.000000
card_id                        0   0.000000
amount                         0   0.000000
use_chip                       0   0.000000
merchant_id                    0   0.000000
merchant_city                  0   0.000000
merchant_state           1563700  11.751916
zip                      1652706  12.420837
mcc                            0   0.000000
errors                  13094522  98.411286


In [8]:
# Điều chỉnh kiểu dữ liệu cho phù hợp
transactions_df['date'] = pd.to_datetime(transactions_df['date'])
transactions_df['amount'] = (transactions_df['amount']
                             .astype(str)
                             .str.replace('$', '', regex=False)
                             .str.replace(',', '', regex=False)
                             .astype(float))

In [9]:
# Kiểm tra outliers
print("Thống kê mô tả chi tiết các cột số:")
display(transactions_df['amount'].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))
print("- Thống kê Outliers toán học bằng phương pháp IQR:")
Q1 = transactions_df['amount'].quantile(0.25)
Q3 = transactions_df['amount'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = transactions_df[(transactions_df['amount'] < lower_bound) | (transactions_df['amount'] > upper_bound)]
print(f"Cột 'amount': Có {outliers.shape[0]} dòng là Outlier (Nằm ngoài khoảng: {lower_bound:.1f} -> {upper_bound:.1f})")

Thống kê mô tả chi tiết các cột số:


count    1.330592e+07
mean     4.297604e+01
std      8.165575e+01
min     -5.000000e+02
1%      -9.600000e+01
5%       0.000000e+00
25%      8.930000e+00
50%      2.899000e+01
75%      6.371000e+01
95%      1.459200e+02
99%      3.159500e+02
max      6.820200e+03
Name: amount, dtype: float64

- Thống kê Outliers toán học bằng phương pháp IQR:
Cột 'amount': Có 1052519 dòng là Outlier (Nằm ngoài khoảng: -73.2 -> 145.9)


In [10]:
# Xử lí missing value
transactions_df['errors'] = transactions_df['errors'].fillna('No Error')
transactions_df['merchant_state'] = transactions_df['merchant_state'].fillna('Unknown')
transactions_df['zip'] = transactions_df['zip'].fillna('Unknown')

In [11]:
mixed_columns = ['zip', 'merchant_state', 'merchant_city', 'errors']

for col in mixed_columns:
    if col in transactions_df.columns:
        transactions_df[col] = transactions_df[col].astype(str)

In [12]:
print(transactions_df.isnull().sum())

id                0
date              0
client_id         0
card_id           0
amount            0
use_chip          0
merchant_id       0
merchant_city     0
merchant_state    0
zip               0
mcc               0
errors            0
dtype: int64


In [13]:
# Dữ liệu sau khi xử lý
transactions_df.head()

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,2010-01-01 00:01:00,1556,2972,-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,No Error
1,7475328,2010-01-01 00:02:00,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,No Error
2,7475329,2010-01-01 00:02:00,1129,102,80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,No Error
3,7475331,2010-01-01 00:05:00,430,2860,200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,No Error
4,7475332,2010-01-01 00:06:00,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,No Error


# Xử lý file 'cards'

In [14]:
# Đọc file
cards_df = pd.read_csv('cards_data.csv')

In [15]:
# Dữ liệu trước khi xử lý
cards_df.head(5)

,id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,4524,825,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
1,2731,825,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
2,3701,825,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
3,42,825,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
4,4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No


In [16]:
# Xem qua số cột và dòng trong df
rows, cols = cards_df.shape
print(f"Kích thước bảng: {rows} dòng và {cols} cột.")

Kích thước bảng: 6146 dòng và 13 cột.


In [17]:
# Tổng quan về kiểu dữ liệu từng cột
cards_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6146 entries, 0 to 6145
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   id                     6146 non-null   int64 
 1   client_id              6146 non-null   int64 
 2   card_brand             6146 non-null   object
 3   card_type              6146 non-null   object
 4   card_number            6146 non-null   int64 
 5   expires                6146 non-null   object
 6   cvv                    6146 non-null   int64 
 7   has_chip               6146 non-null   object
 8   num_cards_issued       6146 non-null   int64 
 9   credit_limit           6146 non-null   object
 10  acct_open_date         6146 non-null   object
 11  year_pin_last_changed  6146 non-null   int64 
 12  card_on_dark_web       6146 non-null   object
dtypes: int64(6), object(7)
memory usage: 624.3+ KB


In [18]:
# Tổng quan các giá trị trùng lặp
cards_duplicate_rows = cards_df.duplicated().sum()
cards_duplicate_ids = cards_df['id'].duplicated().sum()
print(f"Số dòng trùng lặp hoàn toàn: {cards_duplicate_rows}")
print(f"Số ID giao dịch bị trùng lặp: {cards_duplicate_ids}")

Số dòng trùng lặp hoàn toàn: 0
Số ID giao dịch bị trùng lặp: 0


In [19]:
# Tổng quan tình trạng Missing value
cards_missing_data = cards_df.isnull().sum()
cards_missing_percent = (cards_df.isnull().sum() / rows) * 100
cards_missing_df = pd.DataFrame({'Số lượng Missing': cards_missing_data, 'Tỷ lệ (%)': cards_missing_percent})
print("Tình trạng Missing Value theo từng cột:")
print(cards_missing_df[cards_missing_df['Số lượng Missing'] >= 0])

Tình trạng Missing Value theo từng cột:
                       Số lượng Missing  Tỷ lệ (%)
id                                    0        0.0
client_id                             0        0.0
card_brand                            0        0.0
card_type                             0        0.0
card_number                           0        0.0
expires                               0        0.0
cvv                                   0        0.0
has_chip                              0        0.0
num_cards_issued                      0        0.0
credit_limit                          0        0.0
acct_open_date                        0        0.0
year_pin_last_changed                 0        0.0
card_on_dark_web                      0        0.0


In [20]:
# Điều chỉnh kiểu dữ liệu cho phù hợp
cards_df['expires'] = pd.to_datetime(cards_df['expires'], format='%m/%Y')
cards_df['acct_open_date'] = pd.to_datetime(cards_df['acct_open_date'], format='%m/%Y')
cards_df['credit_limit'] = (cards_df['credit_limit']
                             .astype(str)
                             .str.replace('$', '', regex=False)
                             .str.replace(',', '', regex=False)
                             .astype(float))

In [21]:
# Kiểm tra outliers
print("Thống kê mô tả chi tiết các cột số:")
display(cards_df['credit_limit'].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))
limit = cards_df[cards_df['credit_limit'] < 0].shape[0]
print(f"Số thẻ tín dụng có hạn mức bị âm (< 0): {limit}")

print("- Thống kê Outliers toán học bằng phương pháp IQR:")
Q1 = cards_df['credit_limit'].quantile(0.25)
Q3 = cards_df['credit_limit'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = cards_df[(cards_df['credit_limit'] < lower_bound) | (cards_df['credit_limit'] > upper_bound)]
print(f"Cột 'credit_limit': Có {outliers.shape[0]} dòng là Outlier (Nằm ngoài khoảng: {lower_bound:.1f} -> {upper_bound:.1f})")

Thống kê mô tả chi tiết các cột số:


count      6146.000000
mean      14347.493980
std       12014.463884
min           0.000000
1%           20.000000
5%           63.000000
25%        7042.750000
50%       12592.500000
75%       19156.500000
95%       34282.500000
99%       55908.950000
max      151223.000000
Name: credit_limit, dtype: float64

Số thẻ tín dụng có hạn mức bị âm (< 0): 0
- Thống kê Outliers toán học bằng phương pháp IQR:
Cột 'credit_limit': Có 235 dòng là Outlier (Nằm ngoài khoảng: -11127.9 -> 37327.1)


In [22]:
# Ép các cột Yes/No thành dạng nhị phân (1 / 0)
cards_df['has_chip'] = cards_df['has_chip'].str.upper().map({'YES': 1, 'NO': 0})
cards_df['card_on_dark_web'] = cards_df['card_on_dark_web'].str.capitalize().map({'Yes': 1, 'No': 0})

In [23]:
# Dữ liệu sau khi xử lý
cards_df.head()

,id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,4524,825,Visa,Debit,4344676511950444,2022-12-01,623,1,2,24295.0,2002-09-01,2008,0
1,2731,825,Visa,Debit,4956965974959986,2020-12-01,393,1,2,21968.0,2014-04-01,2014,0
2,3701,825,Visa,Debit,4582313478255491,2024-02-01,719,1,2,46414.0,2003-07-01,2004,0
3,42,825,Visa,Credit,4879494103069057,2024-08-01,693,0,1,12400.0,2003-01-01,2012,0
4,4659,825,Mastercard,Debit (Prepaid),5722874738736011,2009-03-01,75,1,1,28.0,2008-09-01,2009,0


# Xử lý file 'mcc_codes'

In [24]:
# Đọc file
with open('mcc_codes.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

mcc_codes_df = pd.DataFrame(
    data.items(),
    columns=['code', 'category']
)

In [25]:
# Dữ liệu trước khi xử lý
mcc_codes_df.head(5)

,code,category
0,5812,Eating Places and Restaurants
1,5541,Service Stations
2,7996,"Amusement Parks, Carnivals, Circuses"
3,5411,"Grocery Stores, Supermarkets"
4,4784,Tolls and Bridge Fees


In [26]:
# Xem qua số cột và dòng trong df
rows, cols = mcc_codes_df.shape
print(f"Kích thước bảng: {rows} dòng và {cols} cột.")

Kích thước bảng: 109 dòng và 2 cột.


In [27]:
# Tổng quan về kiểu dữ liệu từng cột
mcc_codes_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109 entries, 0 to 108
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   code      109 non-null    object
 1   category  109 non-null    object
dtypes: object(2)
memory usage: 1.8+ KB


In [28]:
# Tổng quan các giá trị trùng lặp
mcc_codes_duplicate_rows = mcc_codes_df.duplicated().sum()
mcc_codes_duplicate_ids = mcc_codes_df['code'].duplicated().sum()
print(f"Số dòng trùng lặp hoàn toàn: {mcc_codes_duplicate_rows}")
print(f"Số ID giao dịch bị trùng lặp: {mcc_codes_duplicate_ids}")

Số dòng trùng lặp hoàn toàn: 0
Số ID giao dịch bị trùng lặp: 0


In [29]:
# Tổng quan tình trạng Missing value
mcc_codes_missing_data = mcc_codes_df.isnull().sum()
mcc_codes_missing_percent = (mcc_codes_df.isnull().sum() / rows) * 100
mcc_codes_missing_df = pd.DataFrame({'Số lượng Missing': mcc_codes_missing_data, 'Tỷ lệ (%)': mcc_codes_missing_percent})
print("Tình trạng Missing Value theo từng cột:")
print(mcc_codes_missing_df[mcc_codes_missing_df['Số lượng Missing'] >= 0])

Tình trạng Missing Value theo từng cột:
          Số lượng Missing  Tỷ lệ (%)
code                     0        0.0
category                 0        0.0


In [30]:
# Kiểm tra độ dài code
code_lengths = mcc_codes_df['code'].astype(str).str.len().value_counts()
print("Thống kê độ dài ký tự của cột 'code' (Chuẩn là 4):")
print(code_lengths)

Thống kê độ dài ký tự của cột 'code' (Chuẩn là 4):
code
4    109
Name: count, dtype: int64


In [31]:
# Điều chỉnh kiểu dữ liệu cho phù hợp
mcc_codes_df['code'] = mcc_codes_df['code'].astype(int)

In [32]:
# Dữ liệu sau khi xử lý
mcc_codes_df.head()

,code,category
0,5812,Eating Places and Restaurants
1,5541,Service Stations
2,7996,"Amusement Parks, Carnivals, Circuses"
3,5411,"Grocery Stores, Supermarkets"
4,4784,Tolls and Bridge Fees


# Xử lý file 'train_fraud_labels'

In [33]:
# Đọc file
fraud_labels_df = pd.read_json('train_fraud_labels.json')
fraud_labels_df = fraud_labels_df.reset_index()
fraud_labels_df.columns = ['transaction_id', 'fraud_label']

In [34]:
# Dữ liệu trước khi xử lý
fraud_labels_df.head(5)

,transaction_id,fraud_label
0,10649266,No
1,23410063,No
2,9316588,No
3,12478022,No
4,9558530,No


In [35]:
# Xem qua số cột và dòng trong df
rows, cols = fraud_labels_df.shape
print(f"Kích thước bảng: {rows} dòng và {cols} cột.")

Kích thước bảng: 8914963 dòng và 2 cột.


In [36]:
# Tổng quan về kiểu dữ liệu từng cột
fraud_labels_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8914963 entries, 0 to 8914962
Data columns (total 2 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   transaction_id  int64 
 1   fraud_label     object
dtypes: int64(1), object(1)
memory usage: 136.0+ MB


In [37]:
# Tổng quan các giá trị trùng lặp
fraud_labels_duplicate_rows = fraud_labels_df.duplicated().sum()
fraud_labels_duplicate_ids = fraud_labels_df['transaction_id'].duplicated().sum()
print(f"Số dòng trùng lặp hoàn toàn: {fraud_labels_duplicate_rows}")
print(f"Số ID giao dịch bị trùng lặp: {fraud_labels_duplicate_ids}")

Số dòng trùng lặp hoàn toàn: 0
Số ID giao dịch bị trùng lặp: 0


In [38]:
# Tổng quan tình trạng Missing value
fraud_labels_missing_data = fraud_labels_df.isnull().sum()
fraud_labels_missing_percent = (fraud_labels_df.isnull().sum() / rows) * 100
fraud_labels_missing_df = pd.DataFrame({'Số lượng Missing': fraud_labels_missing_data, 'Tỷ lệ (%)': fraud_labels_missing_percent})
print("Tình trạng Missing Value theo từng cột:")
print(fraud_labels_missing_df[fraud_labels_missing_df['Số lượng Missing'] >= 0])

Tình trạng Missing Value theo từng cột:
                Số lượng Missing  Tỷ lệ (%)
transaction_id                 0        0.0
fraud_label                    0        0.0


In [39]:
# Kiểm tra tỷ lệ phân bổ của nhãn gian lận
label_counts = fraud_labels_df['fraud_label'].value_counts()
label_percent = fraud_labels_df['fraud_label'].value_counts(normalize=True) * 100
distribution_df = pd.DataFrame({'Số lượng': label_counts, 'Tỷ lệ (%)': label_percent})
print("Tỷ lệ phân phối nhãn Gian lận (Fraud vs Non-Fraud):")
print(distribution_df)

Tỷ lệ phân phối nhãn Gian lận (Fraud vs Non-Fraud):
             Số lượng  Tỷ lệ (%)
fraud_label                     
No            8901631  99.850454
Yes             13332   0.149546


In [40]:
# Ép các cột Yes/No thành dạng nhị phân (1 / 0)
fraud_labels_df['fraud_label'] = fraud_labels_df['fraud_label'].str.capitalize().map({'Yes': 1, 'No': 0})

In [41]:
# Dữ liệu sau khi xử lý
fraud_labels_df.head()

,transaction_id,fraud_label
0,10649266,0
1,23410063,0
2,9316588,0
3,12478022,0
4,9558530,0


# Xử lý file 'users_data'

In [42]:
# Đọc file
users_df = pd.read_csv('users_data.csv')

In [43]:
# Dữ liệu trước khi xử lý
users_df.head(5)

,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1


In [44]:
# Xem qua số cột và dòng trong df
rows, cols = users_df.shape
print(f"Kích thước bảng: {rows} dòng và {cols} cột.")

Kích thước bảng: 2000 dòng và 14 cột.


In [45]:
# Tổng quan về kiểu dữ liệu từng cột
users_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 2000 non-null   int64  
 1   current_age        2000 non-null   int64  
 2   retirement_age     2000 non-null   int64  
 3   birth_year         2000 non-null   int64  
 4   birth_month        2000 non-null   int64  
 5   gender             2000 non-null   object 
 6   address            2000 non-null   object 
 7   latitude           2000 non-null   float64
 8   longitude          2000 non-null   float64
 9   per_capita_income  2000 non-null   object 
 10  yearly_income      2000 non-null   object 
 11  total_debt         2000 non-null   object 
 12  credit_score       2000 non-null   int64  
 13  num_credit_cards   2000 non-null   int64  
dtypes: float64(2), int64(7), object(5)
memory usage: 218.9+ KB


In [46]:
# Tổng quan các giá trị trùng lặp
users_duplicate_rows = users_df.duplicated().sum()
users_duplicate_ids = users_df['id'].duplicated().sum()
print(f"Số dòng trùng lặp hoàn toàn: {users_duplicate_rows}")
print(f"Số ID giao dịch bị trùng lặp: {users_duplicate_ids}")

Số dòng trùng lặp hoàn toàn: 0
Số ID giao dịch bị trùng lặp: 0


In [47]:
# Tổng quan tình trạng Missing value
users_missing_data = users_df.isnull().sum()
users_missing_percent = (users_df.isnull().sum() / rows) * 100
users_missing_df = pd.DataFrame({'Số lượng Missing': users_missing_data, 'Tỷ lệ (%)': users_missing_percent})
print("Tình trạng Missing Value theo từng cột:")
print(users_missing_df[users_missing_df['Số lượng Missing'] >= 0])

Tình trạng Missing Value theo từng cột:
                   Số lượng Missing  Tỷ lệ (%)
id                                0        0.0
current_age                       0        0.0
retirement_age                    0        0.0
birth_year                        0        0.0
birth_month                       0        0.0
gender                            0        0.0
address                           0        0.0
latitude                          0        0.0
longitude                         0        0.0
per_capita_income                 0        0.0
yearly_income                     0        0.0
total_debt                        0        0.0
credit_score                      0        0.0
num_credit_cards                  0        0.0


In [48]:
# Điều chỉnh kiểu dữ liệu cho phù hợp
money_cols = ['per_capita_income', 'yearly_income', 'total_debt']
for col in money_cols:
    users_df[col] = (users_df[col]
                                 .astype(str)
                                 .str.replace('$', '', regex=False)
                                 .str.replace(',', '', regex=False)
                                 .astype(float))

In [49]:
# Dữ liệu sau khi xử lý
users_df.head()

,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,29278.0,59696.0,127613.0,787,5
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,37891.0,77254.0,191349.0,701,5
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,22681.0,33483.0,196.0,698,5
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,163145.0,249925.0,202328.0,722,4
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,53797.0,109687.0,183855.0,675,1


In [50]:
# Kiểm tra outliers
numeric_cols = ['current_age', 'retirement_age', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards']
print("Thống kê mô tả chi tiết các cột số:")
display(users_df[numeric_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

print("Kiểm tra các bất thường:")
age_anomaly = users_df[(users_df['current_age'] < 0) | (users_df['current_age'] > 110)].shape[0]
print(f"Số người có tuổi bất thường (<0 hoặc >110): {age_anomaly}")
credit_anomaly = users_df[(users_df['credit_score'] < 300) | (users_df['credit_score'] > 850)].shape[0]
print(f"Số người có điểm tín dụng vượt khung chuẩn (300-850): {credit_anomaly}")
income_anomaly = users_df[users_df['yearly_income'] < 0].shape[0]
print(f"Số người có thu nhập năm bị âm (< 0): {income_anomaly}")

print("Thống kê Outliers toán học bằng phương pháp IQR:")
for col in numeric_cols:
    Q1 = users_df[col].quantile(0.25)
    Q3 = users_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = users_df[(users_df[col] < lower_bound) | (users_df[col] > upper_bound)]
    print(f" - Cột '{col}': Có {outliers.shape[0]} dòng là Outlier (Nằm ngoài khoảng: {lower_bound:.1f} -> {upper_bound:.1f})")

Thống kê mô tả chi tiết các cột số:


,current_age,retirement_age,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
count,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,45.391500,66.237500,23141.928000,45715.882000,63709.694000,709.734500,3.073000
std,18.414092,3.628867,11324.137358,22992.615456,52254.453421,67.221949,1.637379
min,18.000000,50.000000,0.000000,1.000000,0.000000,480.000000,1.000000
1%,18.000000,56.000000,9277.740000,10138.530000,0.000000,519.970000,1.000000
5%,19.000000,60.000000,12683.650000,22849.700000,0.000000,589.000000,1.000000
25%,30.000000,65.000000,16824.500000,32818.500000,23986.750000,681.000000,2.000000
50%,44.000000,66.000000,20581.000000,40744.500000,58251.000000,711.500000,3.000000
75%,58.000000,68.000000,26286.000000,52698.500000,89070.500000,753.000000,4.000000
95%,80.050000,72.000000,41685.400000,82980.500000,153445.300000,815.000000,6.000000


Kiểm tra các bất thường:
Số người có tuổi bất thường (<0 hoặc >110): 0
Số người có điểm tín dụng vượt khung chuẩn (300-850): 0
Số người có thu nhập năm bị âm (< 0): 0
Thống kê Outliers toán học bằng phương pháp IQR:
 - Cột 'current_age': Có 1 dòng là Outlier (Nằm ngoài khoảng: -12.0 -> 100.0)
 - Cột 'retirement_age': Có 210 dòng là Outlier (Nằm ngoài khoảng: 60.5 -> 72.5)
 - Cột 'per_capita_income': Có 123 dòng là Outlier (Nằm ngoài khoảng: 2632.2 -> 40478.2)
 - Cột 'yearly_income': Có 118 dòng là Outlier (Nằm ngoài khoảng: 2998.5 -> 82518.5)
 - Cột 'total_debt': Có 51 dòng là Outlier (Nằm ngoài khoảng: -73638.9 -> 186696.1)
 - Cột 'credit_score': Có 67 dòng là Outlier (Nằm ngoài khoảng: 573.0 -> 861.0)
 - Cột 'num_credit_cards': Có 20 dòng là Outlier (Nằm ngoài khoảng: -1.0 -> 7.0)


# Nối 2 bảng transactions và fraud_labels

In [51]:
final_transactions_df = transactions_df.merge(
    fraud_labels_df[['transaction_id', 'fraud_label']],
    left_on='id',                          
    right_on='transaction_id',
    how='left' 
)

In [52]:
final_transactions_df = final_transactions_df.rename(columns={'fraud_label': 'is_fraud'})

In [53]:
final_transactions_df =final_transactions_df.drop(columns=['transaction_id'])

In [54]:
final_transactions_df['is_fraud'] = final_transactions_df['is_fraud'].fillna(-1)

In [55]:
final_transactions_df['is_fraud'] = final_transactions_df['is_fraud'].astype(int)

In [56]:
final_transactions_df.head()

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,is_fraud
0,7475327,2010-01-01 00:01:00,1556,2972,-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,No Error,0
1,7475328,2010-01-01 00:02:00,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,No Error,0
2,7475329,2010-01-01 00:02:00,1129,102,80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,No Error,0
3,7475331,2010-01-01 00:05:00,430,2860,200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,No Error,-1
4,7475332,2010-01-01 00:06:00,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,No Error,0


# Đẩy dữ liệu lên Big Query

In [57]:
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account

project_id = "jda-k1"
dataset_id = "bin_bigproject"
key_path = r"C:\Users\LENOVO\Downloads\jda-k1-2afad6d8f47e.json"

# Khởi tạo client kết nối BigQuery
credentials = service_account.Credentials.from_service_account_file(key_path)
client = bigquery.Client(credentials=credentials, project=project_id)


tables_pipeline = [
    {"table_id": "dim_mcc_codes", "dataframe": mcc_codes_df},
    {"table_id": "dim_users", "dataframe": users_df},
    {"table_id": "dim_cards", "dataframe": cards_df},
    {"table_id": "fact_transactions", "dataframe": final_transactions_df}  
]


print("Bắt đầu đẩy trực tiếp DataFrame từ Notebook lên BigQuery...\n")

job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    autodetect=True
)

for table_info in tables_pipeline:
    table_id = table_info["table_id"]
    df_current = table_info["dataframe"]
    table_ref = f"{project_id}.{dataset_id}.{table_id}"
    
    print(f"Đang đẩy bảng '{table_id}' ({df_current.shape[0]:,} dòng)...")
    
    try:
        job = client.load_table_from_dataframe(df_current, table_ref, job_config=job_config)
        job.result()
        
        print(f"THÀNH CÔNG: Đã nạp {job.output_rows:,} dòng vào {table_ref}\n")
    except Exception as e:
        print(f"LỖI tại bảng {table_id}. Chi tiết: {e}\n")

print("HOÀN THÀNH PIPELINE!")

Bắt đầu đẩy trực tiếp DataFrame từ Notebook lên BigQuery...

Đang đẩy bảng 'dim_mcc_codes' (109 dòng)...


C:\Users\LENOVO\anaconda3\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


THÀNH CÔNG: Đã nạp 109 dòng vào jda-k1.bin_bigproject.dim_mcc_codes

Đang đẩy bảng 'dim_users' (2,000 dòng)...


C:\Users\LENOVO\anaconda3\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


THÀNH CÔNG: Đã nạp 2,000 dòng vào jda-k1.bin_bigproject.dim_users

Đang đẩy bảng 'dim_cards' (6,146 dòng)...


C:\Users\LENOVO\anaconda3\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


THÀNH CÔNG: Đã nạp 6,146 dòng vào jda-k1.bin_bigproject.dim_cards

Đang đẩy bảng 'fact_transactions' (13,305,915 dòng)...


C:\Users\LENOVO\anaconda3\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


THÀNH CÔNG: Đã nạp 13,305,915 dòng vào jda-k1.bin_bigproject.fact_transactions

HOÀN THÀNH PIPELINE!
